# RelCheck v6 — Viability Test (30 images)
## Multi-Prompt BLIP-2 + Image-Grounded Corrector + GroundingDINO (Spatial) + Llama-4-Maverick VLM (Action/Attribute)

**Key improvement over v2:** Multi-prompt captioning extracts richer relational content from BLIP-2.
- 4 focused prompts (detail, spatial, action, attribute) → merged by Llama-3.3-70B
- BLIP-2 remains sole visual source — Llama only reorganizes text (cross-model verification intact)
- Spatial triples → GroundingDINO bbox geometry (deterministic)
- Action/attribute triples → Llama-4-Maverick VLM + multi-question voting

**Decision gate (Cell 9):** Are corrections sensible on ≥60% of corrected images?
- YES → Build full pipeline for 600-image evaluation
- NO → Evaluate which component failed and pivot

In [ ]:
# ============================================================
# Cell 1 — Install Dependencies + Setup
# ============================================================
# Runtime: ~2 min on Colab T4

!pip install -q together transformers pillow torch torchvision
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, textwrap
from io import BytesIO
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from together import Together
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    AutoProcessor, AutoModelForZeroShotObjectDetection,
)

# ── API Key ──
TOGETHER_API_KEY = ""  # <-- paste your key here
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

# ── Model IDs ──
VLM_MODEL   = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"   # vision verifier
LLM_MODEL   = "meta-llama/Llama-3.3-70B-Instruct-Turbo"             # text-only
GDINO_MODEL = "IDEA-Research/grounding-dino-tiny"                    # detection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:   {DEVICE}")
print(f"Captioner: BLIP-2 (Salesforce/blip2-flan-t5-xl) + multi-prompt")
print(f"VLM:      {VLM_MODEL}")
print(f"LLM:      {LLM_MODEL}")
print(f"GDino:    {GDINO_MODEL}")
print("Setup complete.")

In [ ]:
# ============================================================
# Cell 2 — Load Models (BLIP-2 + GroundingDINO on GPU)
# ============================================================
# Runtime: ~3 min (downloading weights first time)

print("Loading BLIP-2 (captioner)...")
blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float16
).to(DEVICE)
print("  BLIP-2 loaded.")

print("Loading GroundingDINO (detector)...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    GDINO_MODEL
).to(DEVICE)
print("  GroundingDINO loaded.")

print("\nAll local models loaded.")

In [ ]:
# ============================================================
# Cell 3 — Load 30 Test Images (from Google Drive cache)
# ============================================================
# Uses images already downloaded by the Master notebook.
# NO re-downloading — only uses what's already on Drive.

from google.colab import drive
from pathlib import Path
import random

# Mount Google Drive (will prompt if not already mounted)
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

# ── Check what's available ──
if not os.path.isdir(DRIVE_IMAGES_DIR):
    print(f"ERROR: Image cache not found at {DRIVE_IMAGES_DIR}")
    print("Run the Master notebook (Cells 0-22) first to download images.")
    raise FileNotFoundError(DRIVE_IMAGES_DIR)

cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
print(f"Found {len(cached_images)} cached images on Drive.")

if len(cached_images) < 20:
    print(f"WARNING: Only {len(cached_images)} images cached. Need at least 20.")
    print("Run the Master notebook image download cell first.")

# ── Load R-Bench annotations if available (for diverse selection) ──
rbench_loaded = False
if os.path.exists(RBENCH_PATH):
    with open(RBENCH_PATH) as f:
        rbench_data = json.load(f)
    rbench_image_ids = set()
    for entry in rbench_data:
        img_field = entry.get("image", entry.get("image_id", ""))
        if img_field:
            img_id = os.path.splitext(os.path.basename(str(img_field)))[0]
            rbench_image_ids.add(img_id)
    print(f"R-Bench covers {len(rbench_image_ids)} unique images.")
    rbench_loaded = True
else:
    print("R-Bench annotations not found — will select random cached images.")
    rbench_image_ids = set()

# ── Select 30 images: prefer those with R-Bench annotations ──
NUM_IMAGES = 30
random.seed(42)  # reproducible

cached_id_to_path = {}
for p in cached_images:
    img_id = p.stem
    cached_id_to_path[img_id] = str(p)

if rbench_loaded:
    annotated = [iid for iid in cached_id_to_path if iid in rbench_image_ids]
    unannotated = [iid for iid in cached_id_to_path if iid not in rbench_image_ids]
    random.shuffle(annotated)
    random.shuffle(unannotated)
    selected_ids = annotated[:NUM_IMAGES]
    if len(selected_ids) < NUM_IMAGES:
        selected_ids += unannotated[:NUM_IMAGES - len(selected_ids)]
else:
    all_ids = list(cached_id_to_path.keys())
    random.shuffle(all_ids)
    selected_ids = all_ids[:NUM_IMAGES]

print(f"\nSelected {len(selected_ids)} images for viability test.")

# ── Load images ──
images = {}
for img_id in selected_ids:
    path = cached_id_to_path[img_id]
    try:
        images[img_id] = Image.open(path).convert("RGB")
    except Exception as e:
        print(f"  Failed to open {img_id}: {e}")

print(f"Loaded {len(images)} images successfully.")

# Show first 8 as preview
preview_ids = list(images.keys())[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_id in zip(axes.flat, preview_ids):
    ax.imshow(images[img_id])
    ax.set_title(img_id[:12], fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(preview_ids):]:
    ax.axis("off")
plt.suptitle(f"Preview: 8 of {len(images)} test images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 4 — Stage 1: Multi-Prompt BLIP-2 Caption Generation
# ============================================================
# Instead of one generic caption, we ask BLIP-2 multiple focused
# questions to extract spatial, action, and attribute content.
# Llama-3.3-70B merges responses into one coherent caption.
# BLIP-2 remains the sole visual source — Llama only reorganizes text.

CAPTION_PROMPTS = [
    "Describe this image in detail.",
    "Where are the objects located relative to each other in this image?",
    "What are the people or animals doing?",
    "Describe the colors, sizes, and attributes of objects in this image.",
]

MERGE_PROMPT = """Merge these image descriptions into ONE coherent paragraph.
Use ONLY information present in the descriptions — do not add anything new.
Focus on spatial relationships (on, next to, behind, in), actions, and attributes.
Remove any redundancy. Be specific and factual. Keep it under 3 sentences.

Descriptions:
{descriptions}

Merged paragraph:"""

captions = {}
for img_id, img in images.items():
    # Step 1: Multi-prompt BLIP-2 captioning
    sub_captions = []
    for prompt_text in CAPTION_PROMPTS:
        inputs = blip2_processor(images=img, text=prompt_text, return_tensors="pt").to(DEVICE, torch.float16)
        with torch.no_grad():
            out = blip2_model.generate(**inputs, max_new_tokens=80)
        response = blip2_processor.decode(out[0], skip_special_tokens=True).strip()
        if response:
            sub_captions.append(response)

    # Step 2: Merge with Llama (text-only — no visual info added)
    if len(sub_captions) > 1:
        numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(sub_captions))
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": MERGE_PROMPT.format(descriptions=numbered)}],
            temperature=0.0,
            max_tokens=200,
        )
        caption = resp.choices[0].message.content.strip()
        if caption.startswith('"') and caption.endswith('"'):
            caption = caption[1:-1]
        if caption.startswith("'") and caption.endswith("'"):
            caption = caption[1:-1]
        # Strip meta-text leakage from LLM (e.g., "Revised paragraph:", "Merged:", "Here is...")
        import re as _re
        caption = _re.sub(r'^(Revised paragraph|Merged paragraph|Merged|Here is the merged[^:]*)\s*:\s*', '', caption, flags=_re.IGNORECASE).strip()
    else:
        caption = sub_captions[0] if sub_captions else ""

    captions[img_id] = caption
    print(f"[{img_id}]")
    for i, sc in enumerate(sub_captions):
        print(f"  Prompt {i+1}: {sc}")
    print(f"  Merged: {caption[:150]}...")
    print()

# ── Detect degenerate (repetitive) captions ──
def is_degenerate(caption, threshold=0.4):
    """Detect repetition loops. Returns True if caption is degenerate."""
    words = caption.lower().split()
    if len(words) < 6:
        return False
    trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
    counts = Counter(trigrams)
    most_common_count = counts.most_common(1)[0][1]
    return most_common_count > 3

degen_count = 0
for img_id, caption in list(captions.items()):
    if is_degenerate(caption):
        print(f"  DEGENERATE [{img_id}]: {caption[:80]}...")
        del captions[img_id]
        del images[img_id]
        degen_count += 1

print(f"\nGenerated {len(captions)} multi-prompt captions via BLIP-2 + Llama merge.")
if degen_count:
    print(f"({degen_count} degenerate captions removed.)")

In [ ]:
# ============================================================
# Cell 5 — Stage 2: Triple Extraction + Type Classification
# ============================================================

TRIPLE_EXTRACTION_PROMPT = """Extract ALL relational triples from this caption as JSON.

Each triple: {{"subject": "...", "relation": "...", "object": "...", "type": "SPATIAL|ACTION|ATTRIBUTE"}}

Type rules:
- SPATIAL: physical position (on, in, above, below, next to, behind, near, under, between, etc.)
- ACTION: interaction/activity (holding, riding, eating, wearing, sitting on, walking, playing with, using, with, etc.)
- ATTRIBUTE: property/quality (is red, is large, is wooden, looks old, etc.)

IMPORTANT formatting rules:
- Every triple MUST have non-null subject, relation, AND object fields
- For ATTRIBUTE triples, put the property as the object: ("car", "is", "red"), NOT ("car", "is red", null)
- For intransitive actions, use the location/context as object: ("man", "standing", "near the door") or ("man", "standing", "in the image")
- If you truly cannot identify an object, use a contextual phrase like "in the scene"

Caption: "{caption}"

Return ONLY a JSON array of triples. No explanation."""

all_triples = {}
for img_id, caption in captions.items():
    prompt = TRIPLE_EXTRACTION_PROMPT.format(caption=caption)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=1500,
    )
    raw = resp.choices[0].message.content.strip()
    # Parse JSON (handle markdown code blocks)
    json_str = raw
    if "```" in json_str:
        json_str = json_str.split("```")[1]
        if json_str.startswith("json"):
            json_str = json_str[4:]
    # Fix truncated JSON: find last complete object and close the array
    json_str = json_str.strip()
    if json_str and not json_str.endswith("]"):
        last_brace = json_str.rfind("}")
        if last_brace > 0:
            json_str = json_str[:last_brace + 1] + "]"
    try:
        triples = json.loads(json_str.strip())
    except json.JSONDecodeError:
        print(f"  [{img_id}] JSON parse failed, raw: {raw[:200]}")
        triples = []

    # Filter out malformed triples (missing fields or None values)
    triples = [t for t in triples if t.get("subject") and t.get("relation") and t.get("object") and t.get("type")]
    # Filter out compositional/possessive relations that aren't verifiable
    COMPOSITIONAL_RELATIONS = {"of", "has", "has a", "with a", "made of", "is a", "is an"}
    before_filter = len(triples)
    triples = [t for t in triples if t["relation"].lower().strip() not in COMPOSITIONAL_RELATIONS]
    if len(triples) < before_filter:
        print(f"  [{img_id}] Filtered {before_filter - len(triples)} compositional relations")

    # Deduplicate triples (same subject+relation+object)
    seen = set()
    unique_triples = []
    for t in triples:
        key = (t["subject"].lower(), t["relation"].lower(), t["object"].lower())
        if key not in seen:
            seen.add(key)
            unique_triples.append(t)
    if len(triples) != len(unique_triples):
        print(f"  [{img_id}] Deduped: {len(triples)} -> {len(unique_triples)} triples")
    all_triples[img_id] = unique_triples
    print(f"\n[{img_id}] Caption: {caption[:120]}...")
    for t in unique_triples:
        print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> {t['type']}")

total = sum(len(v) for v in all_triples.values())
print(f"\nExtracted triples for {len(all_triples)} images. Total: {total}")

In [ ]:
# ============================================================
# Cell 6 — Stage 3a: GroundingDINO Detection + Spatial Geometry
# ============================================================

DETECTION_THRESHOLD = 0.25

def detect_objects(image, text_queries, threshold=DETECTION_THRESHOLD):
    """Detect objects using GroundingDINO. Returns list of (label, score, bbox_norm)."""
    text = ". ".join(text_queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = gdino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=threshold,
        text_threshold=threshold,
        target_sizes=[image.size[::-1]],
    )[0]

    detections = []
    W, H = image.size
    labels_key = "text_labels" if "text_labels" in results else "labels"
    for score, label, box in zip(results["scores"], results[labels_key], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        bbox_norm = [x1/W, y1/H, x2/W, y2/H]
        detections.append((label, score.item(), bbox_norm))
    return detections


def check_spatial_relation(subj_box, obj_box, relation):
    """
    Check if a spatial relation holds between two bounding boxes.
    Boxes are [x1, y1, x2, y2] normalized to [0, 1].
    Returns: True (supported), False (contradicted), None (can't determine)
    """
    sx1, sy1, sx2, sy2 = subj_box
    ox1, oy1, ox2, oy2 = obj_box

    s_cx, s_cy = (sx1+sx2)/2, (sy1+sy2)/2
    o_cx, o_cy = (ox1+ox2)/2, (oy1+oy2)/2

    ix1, iy1 = max(sx1, ox1), max(sy1, oy1)
    ix2, iy2 = min(sx2, ox2), min(sy2, oy2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_s = max((sx2-sx1) * (sy2-sy1), 1e-8)
    area_o = max((ox2-ox1) * (oy2-oy1), 1e-8)
    union = area_s + area_o - inter
    iou = inter / union if union > 0 else 0
    containment = inter / area_s

    rel = relation.lower().strip()

    if rel in ("on", "on top of", "atop"):
        h_overlap = max(0, min(sx2, ox2) - max(sx1, ox1))
        h_denom = min(sx2-sx1, ox2-ox1)
        h_ratio = h_overlap / h_denom if h_denom > 1e-8 else 0
        if h_ratio < 0.15:
            return False
        subj_bottom_near_obj_top = abs(sy2 - oy1) < 0.15
        centroid_above = s_cy < o_cy
        strong_overlap = iou > 0.1
        # NEW: containment check — small object inside larger surface (plate on table)
        subj_contained = (sx1 >= ox1 - 0.05 and sx2 <= ox2 + 0.05 and
                          sy1 >= oy1 - 0.05 and sy2 <= oy2 + 0.05)
        subj_in_upper_half = s_cy < o_cy + 0.1  # subject in upper region of object
        return ((centroid_above and h_ratio > 0.2) or subj_bottom_near_obj_top or
                strong_overlap or (subj_contained and subj_in_upper_half))

    elif rel in ("above", "over"):
        return s_cy < o_cy

    elif rel in ("below", "under", "beneath", "underneath"):
        return s_cy > o_cy

    elif rel in ("next to", "beside", "near"):
        distance = ((s_cx - o_cx)**2 + (s_cy - o_cy)**2) ** 0.5
        return distance < 0.5 and iou < 0.3

    elif rel in ("in", "inside"):
        return containment > 0.3 or iou > 0.15

    elif rel in ("left of", "to the left of", "to the left"):
        return s_cx < o_cx

    elif rel in ("right of", "to the right of", "to the right"):
        return s_cx > o_cx

    elif rel in ("behind", "in front of", "between"):
        return None

    else:
        return None


def find_best_detection(detections, query):
    """Find the highest-confidence detection matching query."""
    query_lower = query.lower().strip()
    query_words = set(query_lower.split())
    best = None
    best_score = -1

    for label, score, bbox in detections:
        label_lower = label.lower().strip()
        label_words = set(label_lower.split())

        if query_lower == label_lower:
            if score > best_score:
                best = (label, score, bbox)
                best_score = score
            continue

        if query_words & label_words:
            if score > best_score:
                best = (label, score, bbox)
                best_score = score
            continue

        if len(query_lower) > 4 and (query_lower in label_lower or label_lower in query_lower):
            if score > best_score:
                best = (label, score, bbox)
                best_score = score

    return best


# ── Run spatial verification ──
spatial_results = {}
all_detections = {}  # store per-image detections for Cell 7

for img_id, triples in all_triples.items():
    img = images[img_id]
    results = []

    all_queries = set()
    for t in triples:
        if t.get("subject"): all_queries.add(t["subject"].lower())
        if t.get("object"): all_queries.add(t["object"].lower())

    detections = detect_objects(img, list(all_queries))
    all_detections[img_id] = detections  # save for detection-augmented VLM in Cell 7
    print(f"\n[{img_id}] Detected {len(detections)} objects: {[(d[0], round(d[1],2)) for d in detections]}")

    for t in triples:
        if t["type"] != "SPATIAL":
            results.append((t, "SKIP", "Not spatial — will use VLM"))
            continue

        subj_det = find_best_detection(detections, t["subject"])
        obj_det = find_best_detection(detections, t["object"])

        if subj_det is None or obj_det is None:
            missing = []
            if subj_det is None: missing.append(t["subject"])
            if obj_det is None: missing.append(t["object"])
            results.append((t, "FALLBACK_VLM", f"Detection failed for: {missing}"))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> FALLBACK (missing: {missing})")
            continue

        geo_result = check_spatial_relation(subj_det[2], obj_det[2], t["relation"])
        if geo_result is None:
            results.append((t, "FALLBACK_VLM", f"No geometry rule for '{t['relation']}'"))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> FALLBACK (no geo rule)")
        elif geo_result:
            evidence = f"Subj bbox={[round(x,3) for x in subj_det[2]]}, Obj bbox={[round(x,3) for x in obj_det[2]]}. Geometry confirms '{t['relation']}'"
            results.append((t, "SUPPORTED", evidence))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> SUPPORTED (geometry)")
        else:
            evidence = f"Subj bbox={[round(x,3) for x in subj_det[2]]}, Obj bbox={[round(x,3) for x in obj_det[2]]}. Geometry CONTRADICTS '{t['relation']}'"
            results.append((t, "HALLUCINATED", evidence))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> HALLUCINATED (geometry)")

    spatial_results[img_id] = results

# Visualize detections for first image
if images:
    first_id = list(images.keys())[0]
    img = images[first_id]
    all_q = set()
    for t in all_triples[first_id]:
        if t.get("subject"): all_q.add(t["subject"].lower())
        if t.get("object"): all_q.add(t["object"].lower())
    dets = detect_objects(img, list(all_q))

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    W, H = img.size
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(dets), 1)))
    for i, (label, score, bbox) in enumerate(dets):
        x1, y1, x2, y2 = bbox[0]*W, bbox[1]*H, bbox[2]*W, bbox[3]*H
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor=colors[i], facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{label} ({score:.2f})", color=colors[i],
                fontsize=10, fontweight="bold", backgroundcolor="black")
    ax.set_title(f"[{first_id}] GroundingDINO Detections")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Cell 7 — Stage 3b: Detection-Augmented VLM Verification
#          GroundingDINO evidence + Llama-4-Maverick + Multi-Question Voting
# ============================================================
# KEY IMPROVEMENT: VLM questions now include GroundingDINO detection context.
# Instead of just "Is the man riding the horse?", we tell Maverick:
# "GroundingDINO detected man at [bbox] and horse at [bbox]. Based on
#  these detections AND the image, is the man riding the horse?"
# This grounds the VLM's judgment in hard object-level evidence.

def encode_image_base64(image):
    """Convert PIL Image to base64 string."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def format_detection_context(detections, subject, obj):
    """Build a detection evidence string for the VLM prompt."""
    subj_det = find_best_detection(detections, subject)
    obj_det = find_best_detection(detections, obj)

    parts = []
    if subj_det:
        bbox = [round(x, 2) for x in subj_det[2]]
        parts.append(f"'{subject}' detected at bbox {bbox} (conf={subj_det[1]:.2f})")
    else:
        parts.append(f"'{subject}' was NOT detected by GroundingDINO")

    if obj_det:
        bbox = [round(x, 2) for x in obj_det[2]]
        parts.append(f"'{obj}' detected at bbox {bbox} (conf={obj_det[1]:.2f})")
    else:
        parts.append(f"'{obj}' was NOT detected by GroundingDINO")

    return "Object detection evidence: " + "; ".join(parts) + "."


def generate_paraphrases(subject, relation, obj, triple_type, detection_context=""):
    """Generate 3 paraphrased yes/no questions for a triple, with detection context."""
    s, r, o = subject, relation, obj

    # Prefix with detection evidence if available
    prefix = f"{detection_context}\n\n" if detection_context else ""

    if triple_type == "ACTION":
        wearing_words = {"wearing", "dressed in", "putting on"}
        if r.lower().strip() in wearing_words:
            return [
                f"{prefix}Look at the {s} in this image. Is the {s} wearing {o}?",
                f"{prefix}Does the {s} have {o} on?",
                f"{prefix}Can you see {o} on the {s}?",
            ]
        return [
            f"{prefix}Is the {s} {r} the {o}?",
            f"{prefix}Does the {s} appear to be {r} the {o}?",
            f"{prefix}Can you see the {s} {r} the {o} in this image?",
        ]
    elif triple_type == "ATTRIBUTE":
        r_stripped = re.sub(r'^(is|are|was|were|be)\s*', '', r.lower()).strip()
        if r_stripped:
            attr = r_stripped if not o else f"{r_stripped} {o}".strip()
        else:
            attr = o
        wearing_words = {"wearing", "in", "dressed in"}
        if r.lower().strip() in wearing_words or "wear" in r.lower():
            return [
                f"{prefix}Look at the {s} in this image. Is the {s} wearing or dressed in {o}?",
                f"{prefix}Does the {s} have {o} on?",
                f"{prefix}Can you see {o} on the {s}?",
            ]
        return [
            f"{prefix}Is the {s} {attr}?",
            f"{prefix}Does the {s} appear to be {attr}?",
            f"{prefix}Looking at the {s}, would you say it is {attr}?",
        ]
    else:  # spatial fallback or OTHER
        return [
            f"{prefix}Is the {s} {r} the {o}?",
            f"{prefix}In this image, is the {s} {r} the {o}?",
            f"{prefix}Does the image show the {s} {r} the {o}?",
        ]


def vlm_verify_triple(image, questions, max_retries=2):
    """
    Send image + questions to Llama-4-Maverick.
    Returns (yes_ratio, raw_answers).
    """
    b64 = encode_image_base64(image)
    answers = []

    for q in questions:
        prompt = f"{q}\n\nAnswer with ONLY 'yes' or 'no'."
        for attempt in range(max_retries + 1):
            try:
                resp = client.chat.completions.create(
                    model=VLM_MODEL,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {
                                "url": f"data:image/jpeg;base64,{b64}"
                            }},
                        ],
                    }],
                    temperature=0.0,
                    max_tokens=10,
                )
                answer = resp.choices[0].message.content.strip().lower()
                if "yes" in answer:
                    answers.append(1.0)
                elif "no" in answer:
                    answers.append(0.0)
                else:
                    answers.append(0.5)  # ambiguous
                break
            except Exception as e:
                if attempt < max_retries:
                    time.sleep(2)
                else:
                    print(f"    VLM error after {max_retries+1} attempts: {e}")
                    answers.append(0.5)

        time.sleep(0.3)  # rate limit

    yes_ratio = np.mean(answers) if answers else 0.5
    return yes_ratio, answers


# ── Thresholds: Majority Vote with 3 Questions ──
# With 3 binary yes/no questions, possible yes_ratios are discrete:
#   0/3 = 0.00,  1/3 = 0.33,  2/3 = 0.67,  3/3 = 1.00
# SUPPORTED (>= 0.65): majority vote "yes" (2/3 or 3/3)
# HALLUCINATED (< 0.35): majority vote "no" (2/3 or 3/3 no)
# UNCERTAIN [0.35, 0.65): split vote -> no correction applied
VLM_SUPPORTED_THRESHOLD = 0.65
VLM_HALLUCINATED_THRESHOLD = 0.35

# ── Run VLM verification ──
vlm_results = {}

for img_id in all_triples:
    img = images[img_id]
    results = []
    spatial_res = spatial_results.get(img_id, [])
    detections = all_detections.get(img_id, [])

    # Collect triples needing VLM
    needs_vlm = []
    for t, verdict, evidence in spatial_res:
        if verdict in ("SKIP", "FALLBACK_VLM"):
            needs_vlm.append((t, evidence))

    print(f"\n[{img_id}] VLM verification for {len(needs_vlm)} triples (with {len(detections)} detections as context)")

    for t, fallback_reason in needs_vlm:
        # Build detection context for this triple's subject and object
        det_context = format_detection_context(detections, t["subject"], t["object"])
        questions = generate_paraphrases(
            t["subject"], t["relation"], t["object"], t["type"],
            detection_context=det_context
        )
        print(f"    Detection context: {det_context}")
        print(f"    Q1: {questions[0][:120]}...")
        yes_ratio, raw = vlm_verify_triple(img, questions)

        if yes_ratio >= VLM_SUPPORTED_THRESHOLD:
            verdict = "SUPPORTED"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). {det_context} Relation '{t['relation']}' confirmed."
        elif yes_ratio < VLM_HALLUCINATED_THRESHOLD:
            verdict = "HALLUCINATED"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). {det_context} Relation '{t['relation']}' NOT confirmed."
        else:
            verdict = "UNCERTAIN"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). {det_context} Inconclusive."

        results.append((t, verdict, evidence))
        print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> {verdict} (yes_ratio={yes_ratio:.2f})")

    vlm_results[img_id] = results

print("\n=== Detection-Augmented VLM Verification Complete ===")

In [ ]:
# ============================================================
# Cell 8 — Stage 4: Merge All Verdicts + Llama Text-Only Correction + Display
# ============================================================
# KEY LESSON from v6 run: Maverick VLM corrector over-writes captions.
# When given the image, Maverick re-captions instead of making surgical edits.
# Fix: Use Llama-3.3-70B (text-only) as corrector — it can only see the
# structured evidence from Maverick, not the image. This forces minimal edits.
#
# Architecture:
#   Verify:  Maverick sees image → determines WHAT is hallucinated + why
#   Correct: Llama reads evidence → makes minimal word substitutions only

CORRECTION_PROMPT = """You are a precise caption editor making MINIMAL text corrections.

Original caption: "{caption}"

The following relations in the caption have been verified as WRONG by a vision model:
{hallucination_list}

Your task:
1. Change ONLY the specific words describing each hallucinated relation
2. Keep EVERY other word exactly the same — do not rephrase, reorder, or restructure
3. If you must remove a hallucinated claim, delete only that phrase, not the whole sentence
4. Do NOT add new information
5. Output ONLY the corrected caption — no explanation, no prefix

If you cannot fix the hallucinated relation with a small change (< 6 words), remove just that claim.

Corrected caption:"""


def merge_verdicts(img_id):
    """Combine spatial geometry + VLM results into final verdicts."""
    merged = []
    spatial = spatial_results.get(img_id, [])
    vlm = vlm_results.get(img_id, [])

    vlm_lookup = {}
    for t, v, e in vlm:
        key = (t["subject"], t["relation"], t["object"])
        vlm_lookup[key] = (v, e)

    for t, verdict, evidence in spatial:
        key = (t["subject"], t["relation"], t["object"])
        if verdict in ("SUPPORTED", "HALLUCINATED"):
            merged.append((t, verdict, evidence))
        elif verdict in ("SKIP", "FALLBACK_VLM"):
            if key in vlm_lookup:
                v, e = vlm_lookup[key]
                merged.append((t, v, e))
            else:
                merged.append((t, "UNCERTAIN", "No verification available"))
    return merged


def correct_caption_llm(caption, hallucinated_triples):
    """Use Llama-3.3-70B (text-only) to correct hallucinated spans — minimal edits only."""
    if not hallucinated_triples:
        return caption, "No hallucinations found"

    hall_list = ""
    for i, (t, evidence) in enumerate(hallucinated_triples, 1):
        # Pass structured evidence: what Maverick said + detection context
        hall_list += f"{i}. Triple ({t['subject']}, {t['relation']}, {t['object']})\n   Evidence: {evidence}\n"

    prompt = CORRECTION_PROMPT.format(caption=caption, hallucination_list=hall_list)

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,  # Llama-3.3-70B — text only, no image
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=400,
        )
        corrected = resp.choices[0].message.content.strip()
        # Strip any meta-text prefix
        import re as _re
        corrected = _re.sub(r'^(Corrected caption|Here is|Here\'s)[^:]*:\s*', '', corrected, flags=_re.IGNORECASE).strip()
        if corrected.startswith('"') and corrected.endswith('"'):
            corrected = corrected[1:-1]
        return corrected, hall_list
    except Exception as e:
        print(f"  Correction failed: {e}")
        return caption, "Correction failed"


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


# Edit rate gate: if correction changes > 30% of caption, reject it
EDIT_RATE_GATE = 0.30

# ── Run merge + correction ──
final_results = {}

print("=" * 70)
print("RELCHECK V6 — VIABILITY TEST RESULTS (Llama Text-Only Corrector)")
print("=" * 70)

for img_id in images:
    caption = captions[img_id]
    merged = merge_verdicts(img_id)

    hallucinated = [(t, e) for t, v, e in merged if v == "HALLUCINATED"]
    supported = [(t, e) for t, v, e in merged if v == "SUPPORTED"]
    uncertain = [(t, e) for t, v, e in merged if v == "UNCERTAIN"]

    if hallucinated:
        corrected, hall_details = correct_caption_llm(caption, hallucinated)
        edit_rate = levenshtein(caption, corrected) / max(len(caption), 1)
        # Edit rate gate: reject wild rewrites
        if edit_rate > EDIT_RATE_GATE:
            print(f"  [{img_id}] GATE REJECTED edit_rate={edit_rate:.1%} — keeping original")
            corrected = caption
            edit_rate = 0.0
    else:
        corrected = caption
        hall_details = "None"
        edit_rate = 0.0

    final_results[img_id] = {
        "caption": caption,
        "corrected": corrected,
        "triples": len(merged),
        "supported": len(supported),
        "hallucinated": len(hallucinated),
        "uncertain": len(uncertain),
        "edit_rate": edit_rate,
        "details": merged,
    }

# ── Display results with images ──
for img_id, result in final_results.items():
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(images[img_id])
    ax.axis("off")
    ax.set_title(img_id, fontsize=12, fontweight="bold")

    orig = textwrap.fill(f"Original: {result['caption']}", width=90)
    corr = textwrap.fill(f"Corrected: {result['corrected']}", width=90)
    stats = (f"Triples: {result['triples']} | Supported: {result['supported']} | "
             f"Hallucinated: {result['hallucinated']} | Uncertain: {result['uncertain']} | "
             f"Edit rate: {result['edit_rate']:.1%}")

    hall_lines = []
    for item in result["details"]:
        if isinstance(item, (list, tuple)) and len(item) == 3:
            t, verdict, evidence = item
            if verdict == "HALLUCINATED":
                hall_lines.append(f"  X ({t['subject']}, {t['relation']}, {t['object']})")

    caption_text = f"{orig}\n\n{corr}\n\n{stats}"
    if hall_lines:
        caption_text += "\n\nHallucinated:\n" + "\n".join(hall_lines)

    fig.text(0.5, -0.02, caption_text, ha="center", va="top",
             fontsize=9, family="monospace",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))

    plt.tight_layout()
    plt.show()
    print()


In [ ]:
# ============================================================
# Cell 9 — DECISION GATE: Viability Verdict
# ============================================================

print("\n" + "=" * 70)
print("VIABILITY ASSESSMENT — 30 IMAGE TEST (Multi-Prompt BLIP-2 + Llama Corrector)")
print("=" * 70)

total_images = len(final_results)
images_with_hallucinations = sum(1 for r in final_results.values() if r["hallucinated"] > 0)
images_actually_corrected = sum(1 for r in final_results.values()
                                if r["corrected"] != r["caption"])
images_gate_rejected = images_with_hallucinations - images_actually_corrected
images_no_change = total_images - images_with_hallucinations
total_triples = sum(r["triples"] for r in final_results.values())
total_supported = sum(r["supported"] for r in final_results.values())
total_hallucinated = sum(r["hallucinated"] for r in final_results.values())
total_uncertain = sum(r["uncertain"] for r in final_results.values())

corrected_results = [r for r in final_results.values() if r["corrected"] != r["caption"]]
avg_edit = sum(r["edit_rate"] for r in corrected_results) / max(len(corrected_results), 1)
max_edit = max((r["edit_rate"] for r in corrected_results), default=0)
min_edit = min((r["edit_rate"] for r in corrected_results), default=0)

print(f"\n  Captioner:               BLIP-2 (multi-prompt) + Llama text-only corrector")
print(f"  Edit rate gate:          {EDIT_RATE_GATE:.0%} max (rejects wild rewrites)")
print(f"  Images tested:           {total_images}")
print(f"  Images w/ hallucinations:{images_with_hallucinations}/{total_images} ({images_with_hallucinations/max(total_images,1):.0%})")
print(f"  Images actually corrected:{images_actually_corrected}/{total_images} ({images_actually_corrected/max(total_images,1):.0%})")
print(f"  Images gate-rejected:    {images_gate_rejected}")
print(f"  Images unchanged:        {images_no_change}/{total_images}")
print(f"  Total triples:           {total_triples}")
print(f"  Supported:               {total_supported} ({total_supported/max(total_triples,1):.0%})")
print(f"  Hallucinated:            {total_hallucinated} ({total_hallucinated/max(total_triples,1):.0%})")
print(f"  Uncertain:               {total_uncertain} ({total_uncertain/max(total_triples,1):.0%})")
print(f"  Avg triples/image:       {total_triples/max(total_images,1):.1f}")

# ── Verification breakdown ──
geo_sup = sum(1 for r in final_results.values()
              for t, v, e in r["details"] if v == "SUPPORTED" and "Geometry" in e)
geo_hall = sum(1 for r in final_results.values()
               for t, v, e in r["details"] if v == "HALLUCINATED" and "Geometry" in e)
vlm_verified = sum(1 for r in final_results.values()
                   for t, v, e in r["details"] if "VLM" in e or "yes_ratio" in e)
fallback = sum(1 for r in final_results.values()
               for t, v, e in r["details"] if "FALLBACK" in e or "fallback" in e.lower())

print(f"\n--- Verification Strategy Breakdown ---")
print(f"  Geometry SUPPORTED:    {geo_sup}")
print(f"  Geometry HALLUCINATED: {geo_hall}")
print(f"  VLM verified triples:  {vlm_verified}")

print(f"\n--- Correction Quality ---")
if corrected_results:
    print(f"  Images corrected (passed gate): {len(corrected_results)}")
    print(f"  Avg edit rate (corrected only): {avg_edit:.1%}")
    print(f"  Max edit rate: {max_edit:.1%}")
    print(f"  Min edit rate: {min_edit:.1%}")
else:
    print("  No corrections made (all gate-rejected or no hallucinations)")

# ── Example corrections ──
print(f"\n--- Example Corrections (up to 5) ---")
shown = 0
for img_id, r in final_results.items():
    if r["corrected"] != r["caption"] and shown < 5:
        b = r["caption"][:100].replace("\n", " ")
        a = r["corrected"][:100].replace("\n", " ")
        print(f"\n  [{img_id}]")
        print(f"    Before: {b}...")
        print(f"    After:  {a}...")
        print(f"    Edit rate: {r['edit_rate']:.1%} | Hallucinated: {r['hallucinated']}/{r['triples']}")
        shown += 1

# ── Qualitative rating ──
print("\n" + "=" * 70)
print("RATE EACH CORRECTED IMAGE: good (g) / bad (b) / skip (s)")
print("Then enter below for final verdict.")
print("=" * 70)

good, bad = 0, 0
for img_id, r in final_results.items():
    if r["corrected"] != r["caption"]:
        rating = input(f"[{img_id[:8]}] Before: {r['caption'][:60]}...\n"
                       f"  After:  {r['corrected'][:60]}...\n"
                       f"  Rating (g/b/s): ").strip().lower()
        if rating == "g": good += 1
        elif rating == "b": bad += 1

total_rated = good + bad
pct_good = good / max(total_rated, 1)
print(f"\n  Good corrections: {good}/{total_rated} ({pct_good:.0%})")
print(f"  Bad corrections:  {bad}")

print("\n" + "=" * 70)
if pct_good >= 0.75 and images_with_hallucinations >= 8:
    print("  VERDICT: VIABLE — proceed to full 600-image evaluation")
elif pct_good >= 0.60:
    print("  VERDICT: MARGINAL — investigate top bad cases before proceeding")
else:
    print("  VERDICT: NOT VIABLE — investigate: Are VLM answers sensible? False positive rate?")
print("=" * 70)

import json as _json
with open("viability_v6b_results.json", "w") as f:
    _json.dump({k: {**v, "details": [(t, verdict, ev) for t, verdict, ev in v["details"]]}
                for k, v in final_results.items()}, f, indent=2, default=str)
print("\nResults saved to viability_v6b_results.json")
